# Make Database
The point of this notebook is to extract text from the chatgpt answers and create a database that can be used for modelranking.

In [1]:
from pathlib  import Path
from json     import loads, dumps
from datasets import load_dataset

import sqlite3
import re
import pandas as pd

/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset   = load_dataset("arch-be/brabant-xvii", name="doc_by_doc")
responses = list(Path("./gpt-4o-mini").glob("*.txt"))

In [3]:
dataset = pd.concat([
    dataset['train'].to_pandas(),
    dataset['test'].to_pandas(),
    dataset['valid'].to_pandas() ]).set_index(['project', 'file_id'])

In [4]:
db = sqlite3.connect('mistral7b.db')

def get_id_text(proj: str, fid: str) -> tuple[int, str]:
    cur = db.cursor()
    cur.execute("SELECT id, text FROM dataset WHERE project = ? AND file_id = ?;", (proj, fid))
    data = cur.fetchone()
    return data

In [5]:
prompt = Path("./prompt.txt").read_text(encoding="utf8")

def process(path: Path) -> dict:
    if not path.exists():
        return None
    txt = path.read_text(encoding="utf8")
    res = re.search(r"content='(?P<json>.*\})'", txt, flags=re.DOTALL)
    if res:
        res = res['json'].replace(r'\n', '')
        return eval(res)
    else:
        return None

def to_labeling_column(id: int, text: str, obj: dict) -> dict:
    return {
        "id": id,
        "document": text,
        "documentdata": obj,
        "conversation": [
            { "role": "system",    "content": prompt},
            { "role": "user",      "content": text},
            { "role": "assistant", "content": dumps(obj)}
        ]
    }



In [6]:
def row_to_labeling(row: pd.Series) -> dict:
    pid      = row['project']
    fid      = row['file_id']
    gid, txt = get_id_text(pid, fid)
    rspfile  = Path(f"./gpt-4o-mini/{pid}_{fid}")
    if not rspfile.exists():
        return { "id": gid, "document": text }
    else:
        rsp = process(rspfile)
        labeling = to_labeling_column(gid, txt, rsp)
        return labeling

def row_to_slabel(row):
    return dumps(row_to_labeling(row))

In [7]:
dataset = dataset.reset_index()
dataset['labeling'] = dataset.reset_index().apply(row_to_slabel, axis=1)
dataset = dataset.set_index(['project', 'file_id'])
dataset

text  \
project file_id                                                      
pardons 299.txt  kaerle etc doen te wetene allen iegewoirdich e...   
        228.txt  remissie voer\ndiericken gheerits\nkaerle etc....   
        118.txt  philips etc. allen etc. saluit alsoe als tonse...   
        126.txt  philips byder gratien goidts coninck van casti...   
        373.txt  remissie van dootslage\nphilips byder gracien ...   
...                                                            ...   
        012.txt  maximilian par la grace de dieu esleu empereur...   
        099.txt  remissie\nalbert ende isabel clara eugenia etc...   
        315.txt  kaerle etc. doen te wetene allen iegewoirdigen...   
        300.txt  kaerle etc. doen te wetene allen iegewoirdige ...   
        004.txt  kaerle etc. ic doen te wetene allen iegenwoird...   

                                                          labeling  
project file_id                                                     
pardons 299.txt  {"id": 0, "document": "kaerle etc doen te wete...  
        228.txt  {"id": 1, "document": "remissie voer\ndiericke...  
        118.txt  {"id": 2, "document": "philips etc. allen etc....  
        126.txt  {"id": 3, "document": "philips byder gratien g...  
        373.txt  {"id": 4, "document": "remissie van dootslage\...  
...                                                            ...  
        012.txt  {"id": 414, "document": "maximilian par la gra...  
        099.txt  {"id": 415, "document": "remissie\nalbert ende...  
        315.txt  {"id": 416, "document": "kaerle etc. doen te w...  
        300.txt  {"id": 417, "document": "kaerle etc. doen te w...  
        004.txt  {"id": 418, "document": "kaerle etc. ic doen t...  

[419 rows x 2 columns]

In [9]:
con = sqlite3.connect("gpt4o-mini.db")
dataset.reset_index().to_sql("dataset", con=con)

419